<a href="https://colab.research.google.com/github/Suim00/hdac-team5-knhanes/blob/main/10_table1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 10주차 분석 대상 선정과 Table 1 (9팀)

국민건강영양조사 2024년(`hn24_all.sas7bdat`). 연구질문: 우리나라 성인에서 아침 결식은 비만(BMI 25 이상) 유병과 연관되는가?

구성: 1 자료 읽기 → 2 노출·결과·공변량 만들기 → 3 분석 대상 표시변수와 흐름도(B-1) → 4 설계 기반 Table 1(B-2) → 5 비가중 표와 차이(B-3)

## 1. Drive 연결과 변수 정의표의 변수만 읽기

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATA_DIR  = '/content/drive/MyDrive/hdac_data/HN24_ALL(SAS)'
DATA_FILE = f'{DATA_DIR}/hn24_all.sas7bdat'

In [ ]:
!pip -q install pyreadstat
import pyreadstat
import numpy as np
import pandas as pd

# 9주차 정의표의 변수 (원변수명 = 이용지침서 표기)
need_core = ['age', 'sex', 'HE_prg', 'L_BR_FQ', 'HE_ht', 'HE_wt', 'HE_BMI', 'HE_obe',
             'wt_tot', 'kstrata', 'psu']
need_cov  = ['town_t', 'educ', 'sm_presnt', 'dr_month', 'N_DIET', 'N_EN']
need_opt  = ['pa_aerobic']   # 9주차 정의표 하단 '신체활동' 항목. 파일에 있으면 읽어서 코드만 확인 (아래 PA_IN_TABLE 참고)

_, meta0 = pyreadstat.read_sas7bdat(DATA_FILE, metadataonly=True)
lower_map = {c.lower(): c for c in meta0.column_names}
want = need_core + need_cov + need_opt
present = {v: lower_map.get(v.lower()) for v in want}
print('파일에 없는 변수:', [v for v, a in present.items() if a is None] or '없음')

use_cols = [a for a in present.values() if a]
df, meta = pyreadstat.read_sas7bdat(DATA_FILE, usecols=use_cols)
df = df.rename(columns={a: v for v, a in present.items() if a})
print(df.shape)

## 2. 변수 정의표대로 노출·결과·공변량 만들기

원 코드 빈도표를 먼저 출력하고, 같은 셀에서 분석용 변수를 만든다.

In [ ]:
# ---- 노출: 아침식사 빈도 (L_BR_FQ, 이용지침서 218쪽) ----
EXPOSED_CODES   = [3, 4]   # 결식(노출군): 주 1~2회, 거의 안 함
REFERENCE_CODES = [1, 2]   # 비결식(비교군): 주 5~7회, 주 3~4회
assert not set(EXPOSED_CODES) & set(REFERENCE_CODES)

print('L_BR_FQ 원 코드별 인원 (전체 참여자, 결측 포함)')
print(df['L_BR_FQ'].value_counts(dropna=False).sort_index().to_string())

df['breakfast_skip'] = np.where(df['L_BR_FQ'].isin(REFERENCE_CODES), 0.0,
                         np.where(df['L_BR_FQ'].isin(EXPOSED_CODES), 1.0, np.nan))
# 4범주 (보조 표용): 1 주5~7회, 2 주3~4회, 3 주1~2회, 4 거의 안 함
df['bf4'] = df['L_BR_FQ'].where(df['L_BR_FQ'].isin([1, 2, 3, 4]))

print('\nbreakfast_skip (전체 참여자)')
print(df['breakfast_skip'].value_counts(dropna=False).sort_index().to_string())

In [ ]:
# ---- 결과: 비만 (HE_obe 4~6 = BMI 25 이상), BMI (HE_BMI) ----
print('HE_obe 원 코드별 인원 (결측 포함)')
print(df['HE_obe'].value_counts(dropna=False).sort_index().to_string())

df['obese'] = (df['HE_obe'] >= 4).astype(float).where(df['HE_obe'].notna())

# HE_obe 기준 비만과 BMI>=25 기준 비만이 같은지 확인
chk = df['HE_BMI'].notna() & df['HE_obe'].notna()
print('\nHE_obe>=4 와 HE_BMI>=25 가 다른 인원:',
      int(((df.loc[chk, 'HE_BMI'] >= 25) != (df.loc[chk, 'obese'] == 1)).sum()))

In [ ]:
# ---- 공변량: 비해당·무응답 코드를 결측으로 ----
# 9주차 결정: educ 88(비해당)·99(모름/무응답) -> 결측, N_DIET 9(모름/무응답) -> 결측
# town_t, sm_presnt, dr_month: 별도 무응답 코드 없음 (원값 사용)
for v in ['sex', 'town_t', 'educ', 'sm_presnt', 'dr_month', 'N_DIET']:
    print(f'--- {v} 원 코드 (전체 참여자) ---')
    print(df[v].value_counts(dropna=False).sort_index().to_string())
    print()

df['educ_c']   = df['educ'].where(~df['educ'].isin([88, 99]))
df['N_DIET_c'] = df['N_DIET'].where(~df['N_DIET'].isin([9]))

# 선택: 신체활동 (정의표에 넣기로 한 경우에만 Table 1에 포함)
PA_IN_TABLE = False
if 'pa_aerobic' in df.columns:
    print('--- pa_aerobic 원 코드 (확인용) ---')
    print(df['pa_aerobic'].value_counts(dropna=False).sort_index().to_string())

## 3. 분석 대상 표시변수와 단계별 인원 (B-1)

행을 지우지 않고 `in_analysis = 1`로 표시한다. 단계 순서와 기준은 9주차 계획서와 같다.

In [ ]:
cur = pd.Series(True, index=df.index)
flow_rows = []

def step(no, crit, cond, reason, kind):
    global cur
    before = int(cur.sum())
    cur = cur & cond
    after = int(cur.sum())
    flow_rows.append({'단계': no, '기준': crit, '남은 인원': after,
                      '제외 인원': before - after, '제외 이유': reason, '구분': kind})

flow_rows.append({'단계': '0', '기준': '전체 조사 참여자', '남은 인원': int(cur.sum()),
                  '제외 인원': 0, '제외 이유': '출발 대상', '구분': '-'})

step('1', '만 19세 이상', df['age'] >= 19, '만 19세 미만 (대상 연령 밖)', '비해당(대상 아님)')
step('2', '임신 중 아님 (HE_prg != 1)', df['HE_prg'] != 1, '임신 중 (체중·식사 양상 영향)', '대상 제외 기준')
step('3', '아침식사 빈도 응답 (L_BR_FQ 1~4)', df['L_BR_FQ'].isin([1, 2, 3, 4]),
     '노출 정의 불가 (L_BR_FQ 결측)', '결측')

# 이용지침서에 신장·체중 이상값 제외 기준이 없고 팀도 정하지 않았다 (9주차와 같음).
# 따라서 4단계는 결측 제외만 한다. 기준을 정하면 아래 값을 채우면 단계명과 인원이 함께 바뀐다.
BMI_MIN = None
BMI_MAX = None
ok_body = df['HE_ht'].notna() & df['HE_wt'].notna() & df['HE_BMI'].notna()
if BMI_MIN is not None:
    ok_body = ok_body & (df['HE_BMI'] >= BMI_MIN)
if BMI_MAX is not None:
    ok_body = ok_body & (df['HE_BMI'] <= BMI_MAX)
outlier_rule = (BMI_MIN is not None) or (BMI_MAX is not None)
step('4', '신장·체중·BMI 결측 제외' + (' + BMI 이상값 제외' if outlier_rule else ' (이상값 기준 적용 안 함)'),
     ok_body, '결과 정의 불가 (신장·체중·BMI 결측)' + (' 또는 BMI 이상값' if outlier_rule else ''), '결측')

ok_design = (df['wt_tot'].notna() & (df['wt_tot'] > 0) & df['kstrata'].notna() & df['psu'].notna())
step('5', 'wt_tot > 0, kstrata·psu 유효', ok_design, '설계변수 무효', '설계상 제외')

flow = pd.DataFrame(flow_rows)
print(flow.to_string(index=False))

df['in_analysis'] = cur.astype(int)
DOM = (df['in_analysis'] == 1).to_numpy()
print('\n최종 분석 대상 (in_analysis = 1):', int(DOM.sum()), '/ 전체 행 수(삭제 없음):', len(df))
assert df.loc[DOM, 'HE_obe'].notna().all(), '분석 대상 중 HE_obe 결측이 있음'

In [ ]:
# 9주차 인원과 비교
NINE = {'0': 6997, '1': 6033, '2': 6017, '3': 5843, '4': 5749, '5': 5749}
diff = [(r['단계'], NINE[r['단계']], r['남은 인원']) for r in flow_rows if NINE[r['단계']] != r['남은 인원']]
print('9주차와 다른 단계:', diff if diff else '없음 (9주차와 동일)')

In [ ]:
# 단계 3 제외 (노출 결측)의 내용: 영양조사 미참여와 겹치는지
s2 = ((df['age'] >= 19) & (df['HE_prg'] != 1)).to_numpy()
x3 = s2 & ~df['L_BR_FQ'].isin([1, 2, 3, 4]).to_numpy()
print('단계 3 제외 인원:', int(x3.sum()))
print('  이 중 wt_tot 결측:', int(df.loc[x3, 'wt_tot'].isna().sum()))
print('  이 중 HE_BMI 결측:', int(df.loc[x3, 'HE_BMI'].isna().sum()))
print('전체 참여자에서 L_BR_FQ 결측 x wt_tot 결측')
print(pd.crosstab(df['L_BR_FQ'].isna().map({True: 'L_BR_FQ 결측', False: 'L_BR_FQ 응답'}),
                  df['wt_tot'].isna().map({True: 'wt_tot 결측', False: 'wt_tot 있음'})))

In [ ]:
# 단계 4 제외 94명 (신장·체중·BMI 결측)의 내용
b3 = s2 & df['L_BR_FQ'].isin([1, 2, 3, 4]).to_numpy()
x4 = b3 & ~ok_body.to_numpy()
d4 = df[x4]
print('단계 4 제외 인원:', len(d4), '(BMI 이상값 기준 적용 여부:', outlier_rule, ')')
print('  변수별 결측 인원: 신장(HE_ht) %d, 체중(HE_wt) %d, BMI(HE_BMI) %d'
      % (d4['HE_ht'].isna().sum(), d4['HE_wt'].isna().sum(), d4['HE_BMI'].isna().sum()))
print('  신장·체중·BMI 중 하나라도 결측인 인원 = 제외 인원인지:',
      int(d4[['HE_ht', 'HE_wt', 'HE_BMI']].isna().any(axis=1).sum()) == len(d4))
print('  신장·체중 모두 결측:', int((d4['HE_ht'].isna() & d4['HE_wt'].isna()).sum()))
print('  신장만 결측       :', int((d4['HE_ht'].isna() & d4['HE_wt'].notna()).sum()))
print('  체중만 결측       :', int((d4['HE_ht'].notna() & d4['HE_wt'].isna()).sum()))
print('  BMI 결측          :', int(d4['HE_BMI'].isna().sum()), '/ HE_obe 결측:', int(d4['HE_obe'].isna().sum()))
print('  wt_tot 있음       :', int(d4['wt_tot'].notna().sum()))
print('  연령 평균 %.1f, 65세 이상 %d명, 남자 %d명' % (d4['age'].mean(), (d4['age'] >= 65).sum(), (d4['sex'] == 1).sum()))
print('  결식군 %d명, 비결식군 %d명' % ((d4['breakfast_skip'] == 1).sum(), (d4['breakfast_skip'] == 0).sum()))
print('  (비교) 포함된 5,749명의 연령 평균 %.1f, 남자 %d명' % (df.loc[DOM, 'age'].mean(), (df.loc[DOM, 'sex'] == 1).sum()))

In [ ]:
# 설계변수(wt_tot, kstrata, psu)가 이 분석에 맞는지 확인
# wt_tot = 건강설문 + 검진 + 영양조사를 함께 쓸 때의 가중치 (이용지침서 표 6-2, 28쪽)
# 노출(영양조사), 결과(검진), 공변량(건강설문)을 모두 쓰므로 wt_tot이 맞다.
print('wt_tot가 있는 사람 중 L_BR_FQ 결측:', int((df['wt_tot'].notna() & df['L_BR_FQ'].isna()).sum()))
print('wt_tot가 없는 사람 중 L_BR_FQ 응답:', int((df['wt_tot'].isna() & df['L_BR_FQ'].notna()).sum()))
print('분석 대상 중 wt_tot 결측/0 이하:', int(df.loc[DOM, 'wt_tot'].isna().sum()), '/', int((df.loc[DOM, 'wt_tot'] <= 0).sum()))
print('분석 대상 층 수: %d, 집락 수: %d' % (df.loc[DOM, 'kstrata'].nunique(), df.loc[DOM, 'psu'].nunique()))
print('분석 대상 wt_tot 합(추정 모집단, 명): {:,.0f}'.format(df.loc[DOM, 'wt_tot'].sum()))

# 층별 집락 수 (설계 기반 SE에는 층마다 집락이 2개 이상 필요)
npsu = df.groupby(['kstrata', 'psu']).ngroup()
per_stratum = pd.Series(npsu.values).groupby(df['kstrata'].values).nunique()
print('층별 집락 수: 최소 %d, 최대 %d, 집락이 1개뿐인 층 %d개' % (per_stratum.min(), per_stratum.max(), (per_stratum < 2).sum()))

In [ ]:
# 최종 분석 대상의 노출군·비교군
G_TOTAL = ('전체', DOM)
G_SKIP  = ('결식군(노출)', DOM & (df['breakfast_skip'] == 1).to_numpy())
G_REF   = ('비결식군(비교)', DOM & (df['breakfast_skip'] == 0).to_numpy())
GROUPS  = [G_TOTAL, G_SKIP, G_REF]
for name, m in GROUPS:
    print(f'{name}: n = {int(m.sum()):,}')
assert G_SKIP[1].sum() + G_REF[1].sum() == DOM.sum()

# 4범주 (보조)
BF4_LABEL = {1: '주 5~7회', 2: '주 3~4회', 3: '주 1~2회', 4: '거의 안 함'}
GROUPS4 = [G_TOTAL] + [(BF4_LABEL[k], DOM & (df['bf4'] == k).to_numpy()) for k in [1, 2, 3, 4]]
print({n: int(m.sum()) for n, m in GROUPS4})

In [ ]:
# 흐름도 (GitHub에서 보이는 Mermaid). 숫자는 위 flow 출력에서 가져온다
n = {r['단계']: r for r in flow_rows}
f = lambda k: f"{n[k]['남은 인원']:,}"
x = lambda k: f"{n[k]['제외 인원']:,}"
mermaid = f'''```mermaid
flowchart TD
  A0["조사 참여자 전체<br>n = {f('0')}"] --> A1["만 19세 이상<br>n = {f('1')}"]
  A0 -.-> X1["제외: 만 19세 미만 (n = {x('1')})"]
  A1 --> A2["임신 중 아님<br>n = {f('2')}"]
  A1 -.-> X2["제외: 임신 중 (n = {x('2')})"]
  A2 --> A3["아침식사 빈도 응답<br>n = {f('3')}"]
  A2 -.-> X3["제외: 노출 결측 (n = {x('3')})"]
  A3 --> A4["신장·체중·BMI 있음<br>n = {f('4')}"]
  A3 -.-> X4["제외: 신장·체중·BMI 결측 (n = {x('4')})"]
  A4 --> A5["설계변수 유효 = 최종 분석 대상<br>n = {f('5')}"]
  A4 -.-> X5["제외: 설계변수 무효 (n = {x('5')})"]
  A5 --> E["결식군(노출군)<br>n = {int(G_SKIP[1].sum()):,}"]
  A5 --> C["비결식군(비교군)<br>n = {int(G_REF[1].sum()):,}"]
```'''
print(mermaid)

## 4. 설계 기반 Table 1 (B-2)

- 설계: 가중치 `wt_tot`, 층 `kstrata`, 집락 `psu`. 분석 대상(`in_analysis = 1`)은 도메인으로 다루고 행은 지우지 않는다.
- SE: 층·집락을 반영한 Taylor 선형화 분산 (층 안 집락 합의 편차제곱합, 집락이 1개뿐인 층은 제외).
- 결측: 사람을 빼지 않고 해당 변수의 도메인 지시변수에서만 뺀다 (결측값은 0으로 채워 계산해 nan을 막는다).
- 5주차 `svymean_domain`과 같은 계산이지만 팀 코드로 다시 썼다. 5주차 함수를 쓰려면 아래 `svy_mean`을 바꿔 결과가 같은지 비교한다.

In [ ]:
W  = df['wt_tot'].fillna(0).to_numpy(float)          # 가중치 없는 행은 0 (행은 설계에 남김)
CL = df.groupby(['kstrata', 'psu'], sort=False).ngroup().to_numpy()
NCL = CL.max() + 1
CL_STRATA = pd.Series(df['kstrata'].to_numpy()).groupby(CL).first().to_numpy()
STRATA_IDX = [np.where(CL_STRATA == h)[0] for h in np.unique(CL_STRATA)]

def ratio_se(num, den):
    """비 추정량 sum(num)/sum(den)와 Taylor 선형화 SE"""
    D = den.sum()
    R = num.sum() / D
    u = (num - R * den) / D
    uc = np.bincount(CL, weights=u, minlength=NCL)
    var = 0.0
    for idx in STRATA_IDX:
        nh = len(idx)
        if nh > 1:
            var += nh / (nh - 1) * ((uc[idx] - uc[idx].mean()) ** 2).sum()
    return R, np.sqrt(var)

def svy_mean(x, mask):
    """도메인(mask) 안의 가중 평균(비율)과 설계 기반 SE. x의 결측은 도메인에서 빼고 0으로 채운다"""
    xs = pd.Series(x).astype(float)
    ok = mask & xs.notna().to_numpy()
    xv = np.where(ok, xs.fillna(0).to_numpy(), 0.0)
    d = ok.astype(float)
    return ratio_se(W * d * xv, W * d)

def wquantile(v, w, q):
    o = np.argsort(v)
    v, w = v[o], w[o]
    cw = np.cumsum(w)
    return v[min(np.searchsorted(cw, q * cw[-1]), len(v) - 1)]

def wmedian_iqr(x, mask):
    xs = pd.Series(x).astype(float).to_numpy()
    ok = mask & ~np.isnan(xs)
    v, w = xs[ok], W[ok]
    return wquantile(v, w, .5), wquantile(v, w, .25), wquantile(v, w, .75)

# 자기 점검: 전체 도메인 가중 평균(나이)과 SE
m, se = svy_mean(df['age'], DOM)
print('가중 평균 나이 %.2f (SE %.3f)' % (m, se))

In [ ]:
# 연속형 변수의 요약통계 선택: 히스토그램과 왜도
import matplotlib.pyplot as plt
SKEW_CUT   = 1.0               # |왜도| >= 1 이면 중앙값(IQR), 미만이면 평균
FORCE_MEAN = ['HE_BMI']        # 연구계획상 BMI는 가중 평균으로 보고

cont_vars = {'age': 'Age (years)', 'N_EN': 'Energy intake (kcal)', 'HE_BMI': 'BMI (kg/m2)'}
fig, axes = plt.subplots(1, len(cont_vars), figsize=(12, 3))
SKEW = {}
for ax, (v, lab) in zip(axes, cont_vars.items()):
    s = df.loc[DOM, v].dropna()
    SKEW[v] = s.skew()
    ax.hist(s, bins=40)
    ax.set_title(f'{lab}\nskew = {SKEW[v]:.2f}')
plt.tight_layout()
plt.show()
print({v: round(s, 2) for v, s in SKEW.items()})

In [ ]:
# 행 정의: (변수, 표에 쓸 이름, 형태)
COV_SPEC = [
    ('age',       '나이 (세)',              'cont'),
    ('sex',       '성별',                   'cat'),
    ('town_t',    '거주지역 (town_t)',      'cat'),
    ('educ_c',    '교육수준 (educ)',        'cat'),
    ('sm_presnt', '현재흡연 (sm_presnt)',   'cat'),
    ('dr_month',  '음주 (dr_month)',        'cat'),
    ('N_DIET_c',  'N_DIET',                 'cat'),
    ('N_EN',      '총에너지 섭취량 (kcal)', 'cont'),
]
if PA_IN_TABLE:
    COV_SPEC.append(('pa_aerobic', '유산소 신체활동 (pa_aerobic)', 'cat'))
OUT_SPEC = [
    ('obese',  '비만 (BMI 25 이상)',  'cat'),
    ('HE_BMI', 'BMI (kg/m2)',         'cont'),
]
SPEC = COV_SPEC + OUT_SPEC

# 범주 이름: 이용지침서에서 코드 뜻을 확인해 채울 것 (없으면 '코드 값'으로 표시)
LABELS = {
    'sex':   {1: '남자', 2: '여자'},
    'obese': {0: '아니오 (BMI < 25)', 1: '예 (BMI >= 25)'},
    # 'town_t': {1: '...', 2: '...'},
    # 'educ_c': {1: '...', ..., 8: '...'},
    # 'sm_presnt': {0: '...', 1: '...'},
    # 'dr_month':  {0: '...', 1: '...'},
    # 'N_DIET_c':  {1: '...', 2: '...'},
}
def cat_label(var, k):
    return LABELS.get(var, {}).get(k, f'코드 {k:g}')

In [ ]:
PCT_CHECK, SUMMARY_CHOICE = {}, {}

def choose_summary(var):
    if var in FORCE_MEAN or abs(SKEW[var]) < SKEW_CUT:
        return 'mean'
    return 'median'

def build_table(groups, spec, weighted=True, smd_pair=None):
    hdr = [f'{g} (n={int(m.sum()):,})' for g, m in groups]
    out_cols = ['변수'] + hdr + (['SMD'] if smd_pair else [])
    rows = []
    blank = [''] * (len(groups) + (1 if smd_pair else 0))

    for var, lab, typ in spec:
        x = df[var]
        xn = x.to_numpy(float)
        if typ == 'cat':
            rows.append([lab + (', n (가중 %)' if weighted else ', n (%)')] + blank)
            cats = sorted(x[DOM].dropna().unique())
            sums = np.zeros(len(groups))
            smds = []
            for k in cats:
                r = ['  ' + cat_label(var, k)]
                ps = []
                for gi, (g, m) in enumerate(groups):
                    ok = m & ~np.isnan(xn)
                    nk = int(((xn == k) & ok).sum())
                    if weighted:
                        p = svy_mean(np.where(np.isnan(xn), np.nan, (xn == k).astype(float)), m)[0] * 100
                    else:
                        p = nk / ok.sum() * 100
                    sums[gi] += p
                    ps.append(p / 100)
                    r.append(f'{nk:,} ({p:.1f})')
                if smd_pair:
                    p1, p0 = ps[smd_pair[0]], ps[smd_pair[1]]
                    smds.append(abs(p1 - p0) / np.sqrt((p1 * (1 - p1) + p0 * (1 - p0)) / 2))
                    r.append('')
                rows.append(r)
            PCT_CHECK[(var, weighted)] = sums
            if smd_pair:
                rows[-len(cats)][-1] = f'{max(smds):.3f}'   # 범주 중 최대값을 첫 범주 행에 표시
        else:
            summ = choose_summary(var)
            SUMMARY_CHOICE[var] = summ
            if weighted:
                kind = '가중 평균 (SE)' if summ == 'mean' else '가중 중앙값 (IQR)'
            else:
                kind = '평균 (SD)' if summ == 'mean' else '중앙값 (IQR)'
            r = [f'{lab}, {kind}']
            mus, vs = [], []
            for g, m in groups:
                ok = m & ~np.isnan(xn)
                if weighted and summ == 'mean':
                    mu, se = svy_mean(x, m)
                    r.append(f'{mu:.1f} ({se:.2f})')
                elif weighted:
                    md_, q1, q3 = wmedian_iqr(x, m)
                    r.append(f'{md_:.1f} ({q1:.1f}\u2013{q3:.1f})')
                elif summ == 'mean':
                    r.append(f'{xn[ok].mean():.1f} ({xn[ok].std(ddof=1):.2f})')
                else:
                    q1, md_, q3 = np.percentile(xn[ok], [25, 50, 75])
                    r.append(f'{md_:.1f} ({q1:.1f}\u2013{q3:.1f})')
                w_ = W[ok]
                mu_w = np.average(xn[ok], weights=w_)
                mus.append(mu_w)
                vs.append(np.average((xn[ok] - mu_w) ** 2, weights=w_))
            if smd_pair:
                i, j = smd_pair
                r.append(f'{abs(mus[i] - mus[j]) / np.sqrt((vs[i] + vs[j]) / 2):.3f}')
            rows.append(r)

        # 결측 행
        miss = [int((m & np.isnan(xn)).sum()) for g, m in groups]
        if sum(miss) > 0:
            rows.append(['  결측, n'] + [f'{v:,}' for v in miss] + ([''] if smd_pair else []))

    return pd.DataFrame(rows, columns=out_cols)

def to_md(t):
    lines = ['| ' + ' | '.join(t.columns) + ' |', '|' + '---|' * len(t.columns)]
    for _, r in t.iterrows():
        lines.append('| ' + ' | '.join(str(v).replace(' ', '&nbsp;', 2) if str(v).startswith('  ') else str(v) for v in r) + ' |')
    return '\n'.join(lines)

In [ ]:
# Table 1 (설계 기반): 전체 / 결식군 / 비결식군
table1 = build_table(GROUPS, SPEC, weighted=True, smd_pair=(1, 2))
print(table1.to_string(index=False))

In [ ]:
# 범주 비율의 합이 100%인지 확인 (결측은 분모에서 빠지므로 각 열의 합이 100이어야 함)
for (v, wt), s in PCT_CHECK.items():
    if wt:
        print(f'{v:10s}', np.round(s, 3), 'OK' if np.allclose(s, 100) else '<-- 100이 아님')
print('\n연속형 요약통계 선택:', SUMMARY_CHOICE)

In [ ]:
# 공변량·결과 결측 인원 (분석 대상 안)
miss_tab = pd.DataFrame({v: [int((m & df[v].isna().to_numpy()).sum()) for _, m in GROUPS] for v, _, _ in SPEC},
                        index=[g for g, _ in GROUPS]).T
print(miss_tab)
cov_cols = [v for v, _, _ in COV_SPEC]
cc = df.loc[DOM, cov_cols].notna().all(axis=1)
print('\n공변량이 모두 있는 인원(완전자료):', int(cc.sum()), '/ 하나라도 결측:', int((~cc).sum()))
# Table 1은 5,749명 전체(in_analysis = 1)로 만든다. 아래 표시변수는 보정 회귀(11주차)에서
# 공변량 결측 처리에 따라 대상자가 달라질 수 있음을 보여 주는 용도이며, in_analysis를 바꾸지 않는다.
df['complete_case'] = 0
df.loc[cc[cc].index, 'complete_case'] = 1
print('complete_case = 1:', int(df['complete_case'].sum()))

In [ ]:
# 결측 10% 초과 변수 확인용 (해당 없으면 출력 없음)
for v, lab, _ in SPEC:
    r = df.loc[DOM, v].isna().mean()
    if r > 0.10:
        print(f'{v}: 결측 {r:.1%} -> 결측자와 응답자의 특성 비교 필요')
print('결측 10% 초과 점검 끝')

In [ ]:
# 보조: 4범주 열 (SMD 없음)
table1_4 = build_table(GROUPS4, SPEC, weighted=True)
print(table1_4.to_string(index=False))

In [ ]:
# 아침식사 빈도(4범주·이분형)별 비만 비율: 비가중 기술통계 vs 가중 추정치
# 비가중은 설명용이다. 관련성은 가중치·복합표본설계와 보정을 적용한 분석으로 해석한다.
def obese_by(groups):
    rows = []
    for g, m in groups:
        if g == '전체':
            continue
        ok = m & df['obese'].notna().to_numpy()
        n_ob = int((ok & (df['obese'] == 1).to_numpy()).sum())
        pw, se = svy_mean(df['obese'], m)
        rows.append([g, int(ok.sum()), n_ob, 100 * n_ob / ok.sum(), 100 * pw, 100 * se])
    return pd.DataFrame(rows, columns=['집단', 'n', '비만 n', '비가중 %', '가중 %', '가중 % SE'])

print('4범주'); print(obese_by(GROUPS4).round(1).to_string(index=False))
print('\n이분형'); print(obese_by(GROUPS).round(1).to_string(index=False))

In [ ]:
# md에 붙여 넣을 형식 (숫자는 위 출력과 같음)
print(to_md(table1))

## 5. 같은 형식의 비가중 표와 차이 계산 (B-3)

비교 기준은 모든 변수에서 **표준화 차이 = (가중 − 비가중) / 비가중 SD**(범주형은 범주 지시변수의 SD)로 통일하고, 범주형은 %p 차이도 함께 본다.

In [ ]:
table1_u = build_table(GROUPS, SPEC, weighted=False)
print(table1_u.to_string(index=False))

In [ ]:
def compare_wu(spec, mask=DOM):
    rows = []
    for var, lab, typ in spec:
        x = df[var]
        xn = x.to_numpy(float)
        ok = mask & ~np.isnan(xn)
        if typ == 'cat':
            for k in sorted(x[DOM].dropna().unique()):
                pu = ((xn == k) & ok).sum() / ok.sum()
                pw = svy_mean(np.where(np.isnan(xn), np.nan, (xn == k).astype(float)), mask)[0]
                rows.append([var, lab, cat_label(var, k), pu * 100, pw * 100,
                             (pw - pu) * 100, (pw - pu) / np.sqrt(pu * (1 - pu))])
        else:
            mu, sd = xn[ok].mean(), xn[ok].std(ddof=1)
            mw = svy_mean(x, mask)[0]
            rows.append([var, lab, '평균', mu, mw, mw - mu, (mw - mu) / sd])
    c = pd.DataFrame(rows, columns=['변수', '이름', '범주/통계', '비가중', '가중', '차이(%p 또는 단위)', '표준화 차이'])
    c['|표준화 차이|'] = c['표준화 차이'].abs()
    return c

cmp_all = compare_wu(SPEC)
print(cmp_all.round(3).drop(columns='|표준화 차이|').to_string(index=False))

In [ ]:
# 변수별 최대 |표준화 차이| 순위와 상위 3개
rank = (cmp_all.sort_values('|표준화 차이|', ascending=False)
        .drop_duplicates('변수').reset_index(drop=True))
print(rank[['변수', '이름', '범주/통계', '비가중', '가중', '차이(%p 또는 단위)', '표준화 차이']].round(3).to_string(index=False))
TOP3 = rank['변수'].head(3).tolist()
print('\n차이가 큰 변수 3개:', TOP3)

In [ ]:
# 최종 숫자 대조 (문서에 옮긴 숫자는 모두 위 출력에서 가져온다)
assert flow_rows[-1]['남은 인원'] == int(DOM.sum()) == int(G_SKIP[1].sum() + G_REF[1].sum())
print('흐름도 최종 인원 =', int(DOM.sum()), '= 결식군', int(G_SKIP[1].sum()), '+ 비결식군', int(G_REF[1].sum()))
print('Table 1 열 머리 n:', [f'{g}={int(m.sum())}' for g, m in GROUPS])